In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

current_folder = Path.cwd()

if current_folder.name == "notebooks":
    project_root = current_folder.parent
else:
    project_root = current_folder

data_file = (
    project_root
    / "data"
    / "processed"
    / "multi_patient_50_split.csv"
)

df = pd.read_csv(data_file)

print("Shape:", df.shape)
print(df["split"].value_counts())

Shape: (12515, 24)
split
train         8789
test          1866
validation    1860
Name: count, dtype: int64


In [2]:
val_df = df[
    df["split"] == "validation"
].copy()

analysis_df = val_df.dropna(
    subset=[
        "spo2",
        "future_spo2_5min"
    ]
).copy()

print(
    "Validation rows for analysis:",
    len(analysis_df)
)

Validation rows for analysis: 1748


In [3]:
analysis_df["spo2_change_5min"] = (
    analysis_df["future_spo2_5min"]
    - analysis_df["spo2"]
)

analysis_df["abs_spo2_change_5min"] = (
    analysis_df["spo2_change_5min"].abs()
)

analysis_df[
    "abs_spo2_change_5min"
].describe()

count    1748.000000
mean        0.266876
std         1.636072
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max        37.000000
Name: abs_spo2_change_5min, dtype: float64

In [4]:
analysis_df["change_group"] = pd.cut(
    analysis_df["abs_spo2_change_5min"],
    bins=[
        -np.inf,
        1,
        3,
        np.inf
    ],
    labels=[
        "Stable (<1 point)",
        "Moderate (1 to <3 points)",
        "Large (3+ points)"
    ],
    right=False
)

change_counts = (
    analysis_df["change_group"]
    .value_counts()
    .sort_index()
)

change_counts

change_group
Stable (<1 point)            1559
Moderate (1 to <3 points)     144
Large (3+ points)              45
Name: count, dtype: int64

In [5]:
change_percent = (
    analysis_df["change_group"]
    .value_counts(
        normalize=True
    )
    .sort_index()
    * 100
)

change_percent.round(2)

change_group
Stable (<1 point)            89.19
Moderate (1 to <3 points)     8.24
Large (3+ points)             2.57
Name: proportion, dtype: float64

In [6]:
analysis_df[
    [
        "caseid",
        "minute",
        "spo2",
        "future_spo2_5min",
        "spo2_change_5min",
        "abs_spo2_change_5min"
    ]
].sort_values(
    "abs_spo2_change_5min",
    ascending=False
).head(20)

,caseid,minute,spo2,future_spo2_5min,spo2_change_5min,abs_spo2_change_5min
618,7,1,62.0,99.0,37.0,37.0
11331,74,321,100.0,68.0,-32.0,32.0
619,7,2,73.0,100.0,27.0,27.0
3687,22,229,100.0,83.0,-17.0,17.0
865,7,248,100.0,84.5,-15.5,15.5
3683,22,225,100.0,85.0,-15.0,15.0
11336,74,326,68.0,80.0,12.0,12.0
3458,22,0,94.0,100.0,6.0,6.0
4663,28,131,93.0,99.0,6.0,6.0
4608,28,76,100.0,94.0,-6.0,6.0


In [7]:
summary_table = pd.DataFrame({
    "Count": change_counts,
    "Percent": change_percent.round(2)
})

summary_table

,Count,Percent
change_group,,
Stable (<1 point),1559,89.19
Moderate (1 to <3 points),144,8.24
Large (3+ points),45,2.57


In [8]:
spo2_features = [
    "spo2",
    "spo2_lag_1",
    "spo2_lag_3",
    "spo2_lag_5",
    "spo2_change_1min",
    "spo2_change_3min",
    "spo2_rolling_mean_5",
    "spo2_rolling_std_5"
]

spo2_hr_features = spo2_features + [
    "heart_rate",
    "heart_rate_lag_1",
    "heart_rate_rolling_mean_5"
]

target = "future_spo2_5min"

In [9]:
train_df = df[
    df["split"] == "train"
].copy()

train_ready = train_df.dropna(
    subset=[target]
).copy()

print("Training rows:", len(train_ready))
print("Analysis rows:", len(analysis_df))

Training rows: 8310
Analysis rows: 1748


In [10]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import GradientBoostingRegressor

In [11]:
ridge_model = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        ),
        (
            "model",
            Ridge(alpha=1.0)
        )
    ]
)

ridge_model.fit(
    train_ready[spo2_features],
    train_ready[target]
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('scaler', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](8,)","['spo2','spo2_lag_1','spo2_lag_3',...,'spo2_change_3min', 'spo2_rolling_mean_5','spo2_rolling_std_5']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,8
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for 

In [12]:
gb_model = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "model",
            GradientBoostingRegressor(
                n_estimators=200,
                learning_rate=0.05,
                max_depth=3,
                min_samples_leaf=3,
                random_state=42
            )
        )
    ]
)

gb_model.fit(
    train_ready[spo2_hr_features],
    train_ready[target]
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](11,)","['spo2','spo2_lag_1','spo2_lag_3',...,'heart_rate','heart_rate_lag_1', 'heart_rate_rolling_mean_5']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,11
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for string

In [13]:
analysis_df["pred_persistence"] = (
    analysis_df["spo2"]
)

analysis_df["pred_ridge"] = (
    ridge_model.predict(
        analysis_df[spo2_features]
    )
)

analysis_df["pred_gradient_boosting"] = (
    gb_model.predict(
        analysis_df[spo2_hr_features]
    )
)

In [14]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)

def calculate_group_metrics(
    data,
    prediction_column,
    model_name
):

    rows = []

    for group_name, group_df in data.groupby(
        "change_group",
        observed=True
    ):

        actual = group_df[target]
        predicted = group_df[prediction_column]

        mae = mean_absolute_error(
            actual,
            predicted
        )

        rmse = np.sqrt(
            mean_squared_error(
                actual,
                predicted
            )
        )

        rows.append({
            "Model": model_name,
            "Change Group": group_name,
            "Rows": len(group_df),
            "MAE": mae,
            "RMSE": rmse
        })

    return pd.DataFrame(rows)

In [15]:
persistence_group_results = (
    calculate_group_metrics(
        analysis_df,
        "pred_persistence",
        "Persistence Baseline"
    )
)

ridge_group_results = (
    calculate_group_metrics(
        analysis_df,
        "pred_ridge",
        "Ridge - SpO2 History"
    )
)

gb_group_results = (
    calculate_group_metrics(
        analysis_df,
        "pred_gradient_boosting",
        "Gradient Boosting - SpO2 + HR"
    )
)

In [16]:
group_results = pd.concat(
    [
        persistence_group_results,
        ridge_group_results,
        gb_group_results
    ],
    ignore_index=True
)

group_results

,Model,Change Group,Rows,MAE,RMSE
0,Persistence Baseline,Stable (<1 point),1559,0.000962,0.021933
1,Persistence Baseline,Moderate (1 to <3 points),144,1.166667,1.228990
2,Persistence Baseline,Large (3+ points),45,6.600000,10.091250
3,Ridge - SpO2 History,Stable (<1 point),1559,0.228074,0.283933
4,Ridge - SpO2 History,Moderate (1 to <3 points),144,1.018922,1.360777
5,Ridge - SpO2 History,Large (3+ points),45,5.170358,7.964275
6,Gradient Boosting - SpO2 + HR,Stable (<1 point),1559,0.250527,0.392186
7,Gradient Boosting - SpO2 + HR,Moderate (1 to <3 points),144,1.047629,1.324641
8,Gradient Boosting - SpO2 + HR,Large (3+ points),45,5.119594,7.729437


In [17]:
group_results.sort_values(
    [
        "Change Group",
        "MAE"
    ]
)

,Model,Change Group,Rows,MAE,RMSE
8,Gradient Boosting - SpO2 + HR,Large (3+ points),45,5.119594,7.729437
5,Ridge - SpO2 History,Large (3+ points),45,5.170358,7.964275
2,Persistence Baseline,Large (3+ points),45,6.600000,10.091250
4,Ridge - SpO2 History,Moderate (1 to <3 points),144,1.018922,1.360777
7,Gradient Boosting - SpO2 + HR,Moderate (1 to <3 points),144,1.047629,1.324641
1,Persistence Baseline,Moderate (1 to <3 points),144,1.166667,1.228990
0,Persistence Baseline,Stable (<1 point),1559,0.000962,0.021933
3,Ridge - SpO2 History,Stable (<1 point),1559,0.228074,0.283933
6,Gradient Boosting - SpO2 + HR,Stable (<1 point),1559,0.250527,0.392186


In [18]:
group_results_file = (
    project_root
    / "data"
    / "processed"
    / "subgroup_error_results.csv"
)

group_results.to_csv(
    group_results_file,
    index=False
)

print("Subgroup error results saved!")

Subgroup error results saved!
